# PMT Research — paper-figure starter notebook

Pulls the analysis endpoints into pandas DataFrames so the SIGCSE papers' figures
(coverage heatmaps, credit-loss decomposition, choice cost, category gaps,
curricular complexity, time-to-degree) can be reproduced on our dataset.

**Auth:** the API requires your Firebase ID token. Easiest way to grab one: open the
research console in the browser, DevTools → Network → any `/analysis` request →
copy the `Authorization: Bearer …` value. Tokens expire after ~1 h; re-copy when
you get 401s.

Every response carries `dataset_version` — record it next to any figure you keep.

In [ ]:
import pandas as pd
import requests

BASE_URL = 'http://localhost:3000'   # or the deployed research API
TOKEN = 'PASTE-YOUR-ID-TOKEN'
MAJOR = 'computer science'

S = requests.Session()
S.headers['Authorization'] = f'Bearer {TOKEN}'

def fetch(path, **params):
    r = S.get(f'{BASE_URL}{path}', params=params)
    r.raise_for_status()
    data = r.json()
    print(f"{path}: {data.get('n', len(data.get('rows', [])))} rows @ dataset {data.get('dataset_version')}")
    return pd.DataFrame(data['rows'])

## Coverage — the CC × campus articulation heatmap (both papers' core figure)

In [ ]:
coverage = fetch('/analysis/coverage', majorContains=MAJOR)
heat = coverage.pivot_table(index='community_college', columns='school',
                            values='pct_articulated', aggfunc='mean')
heat.style.background_gradient(cmap='RdYlGn', vmin=0, vmax=100).format('{:.0f}%')

## Credit loss — minimal CC course sets, many-to-one, quarter/semester normalization

In [ ]:
loss = fetch('/analysis/credit-loss', majorContains=MAJOR)
(loss.groupby('school')
     .agg(avg_cc_courses=('min_cc_courses', 'mean'),
          avg_semester_equiv=('semester_equiv_required', 'mean'),
          avg_many_to_one=('many_to_one', 'mean'),
          pct_ccs_blocked=('receivers_blocked', lambda s: (s > 0).mean() * 100))
     .round(2))

## Choice cost — courses added by each additional campus (inter-institution misalignment)
Pass an ordered `schoolIds` list; iterate permutations for the papers' averaged 2nd/3rd/4th-choice bars.

In [ ]:
SCHOOL_IDS = sorted(coverage['school_id'].unique().tolist())
choice = fetch('/analysis/choice-cost', majorContains=MAJOR,
               schoolIds=','.join(map(str, SCHOOL_IDS)))
steps = choice.explode('steps').reset_index(drop=True)
steps = pd.concat([steps.drop(columns=['steps']), pd.json_normalize(steps['steps'])], axis=1)
steps.groupby('school')['additional_courses'].mean().round(2)

## Category gaps — % of CCs missing articulation per campus × canonical course category

In [ ]:
gaps = fetch('/analysis/category-gaps', majorContains=MAJOR)
gaps.pivot_table(index='category', columns='school', values='pct_missing')

## Curricular complexity + time-to-degree (need curated prereqs / ADT docs)

In [ ]:
complexity = fetch('/analysis/complexity', majorContains=MAJOR)
ttd = fetch('/analysis/time-to-degree', majorContains=MAJOR)
display(complexity[['school', 'community_college', 'n_courses', 'complexity',
                    'max_delay', 'prereq_data_coverage_pct']].head(10))
ttd.head(10)